# Week 1 — Data Exploration Notebook

**Fraud AI Investigator** — MENA Fintech Portfolio Project

This notebook explores the synthetic fraud dataset generated in Week 1.
The goal is to understand the data distribution before building the alert engine in Week 2.

---

### Before running
Make sure you've generated the data:
```bash
uv run python scripts/generate_data.py
```

Install notebook dependencies:
```bash
uv add --optional notebooks jupyter pandas matplotlib seaborn
```

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('Libraries loaded ✓')

In [ ]:
# ── Load datasets ──────────────────────────────────────────────────────────────

DATA_DIR = Path('../app/data')

with open(DATA_DIR / 'transactions.json') as f:
    raw_txs = json.load(f)

with open(DATA_DIR / 'kyc_profiles.json') as f:
    raw_kyc = json.load(f)

with open(DATA_DIR / 'sanctions_watchlist.json') as f:
    raw_sanctions = json.load(f)

txs = pd.DataFrame(raw_txs)
kyc = pd.DataFrame(raw_kyc)

txs['timestamp'] = pd.to_datetime(txs['timestamp'])
txs['hour'] = txs['timestamp'].dt.hour

print(f'Transactions:  {len(txs)} rows')
print(f'KYC Profiles:  {len(kyc)} rows')
print(f'Sanctions:     {len(raw_sanctions)} entries')

## 1. Dataset overview

In [ ]:
print('=== Transaction schema ===')
print(txs.dtypes)
print('\n=== First 5 transactions ===')
txs.head()

In [ ]:
print('=== Fraud label distribution ===')
print(txs['is_flagged'].value_counts())
print(f'\nFraud rate: {txs["is_flagged"].mean():.1%}')

## 2. Amount distribution — normal vs suspicious

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram: normal transactions
normal = txs[txs['is_flagged'] == False]
axes[0].hist(normal['amount_aed'], bins=20, color='#4CAF50', alpha=0.8, edgecolor='white')
axes[0].set_title('Normal transactions — amount distribution (AED)', fontsize=12)
axes[0].set_xlabel('Amount (AED)')
axes[0].set_ylabel('Count')
axes[0].axvline(x=40000, color='red', linestyle='--', alpha=0.7, label='AED 40k threshold')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# Histogram: suspicious transactions
suspicious = txs[txs['is_flagged'] == True]
axes[1].hist(suspicious['amount_aed'], bins=10, color='#F44336', alpha=0.8, edgecolor='white')
axes[1].set_title('Suspicious transactions — amount distribution (AED)', fontsize=12)
axes[1].set_xlabel('Amount (AED)')
axes[1].set_ylabel('Count')
axes[1].axvline(x=40000, color='red', linestyle='--', alpha=0.7, label='AED 40k threshold')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.suptitle('Transaction Amounts: Normal vs Suspicious', y=1.02, fontsize=14, fontweight='bold')
plt.savefig('../doc/Screenshots/01_amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to doc/Screenshots/')

## 3. Country distribution — where do transactions originate?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

HIGH_RISK = {'IR', 'KP', 'SY', 'MM', 'YE', 'SD'}

for ax, label, df, color in [
    (axes[0], 'Normal', normal, '#4CAF50'),
    (axes[1], 'Suspicious', suspicious, '#F44336'),
]:
    counts = df['country'].value_counts()
    colors = ['#F44336' if c in HIGH_RISK else '#2196F3' for c in counts.index]
    ax.bar(counts.index, counts.values, color=colors, alpha=0.85, edgecolor='white')
    ax.set_title(f'{label} transactions — country of origin', fontsize=12)
    ax.set_xlabel('Country (ISO 3166-1)')
    ax.set_ylabel('Count')

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#F44336', label='High-risk (FATF/UN)'),
        Patch(facecolor='#2196F3', label='Standard jurisdiction'),
    ]
    ax.legend(handles=legend_elements)

plt.tight_layout()
plt.suptitle('Transaction Origins: Country Risk Profile', y=1.02, fontsize=14, fontweight='bold')
plt.savefig('../doc/Screenshots/02_country_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Time-of-day pattern — do fraud transactions cluster at specific hours?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

normal_hours = normal.groupby('hour').size()
suspicious_hours = suspicious.groupby('hour').size()

all_hours = range(24)
normal_counts = [normal_hours.get(h, 0) for h in all_hours]
suspicious_counts = [suspicious_hours.get(h, 0) for h in all_hours]

ax.bar(all_hours, normal_counts, label='Normal', color='#4CAF50', alpha=0.7)
ax.bar(all_hours, suspicious_counts, bottom=normal_counts, label='Suspicious', color='#F44336', alpha=0.8)

ax.set_title('Transaction volume by hour of day (UTC)', fontsize=12)
ax.set_xlabel('Hour (UTC)')
ax.set_ylabel('Transaction count')
ax.set_xticks(all_hours)
ax.legend()
ax.axvspan(2, 5, alpha=0.08, color='red', label='Fraud cluster window')

plt.tight_layout()
plt.savefig('../doc/Screenshots/03_time_of_day.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. KYC profile analysis

In [ ]:
print('=== KYC Profile summary ===')
print(kyc[['customer_id', 'nationality', 'account_age_days', 'has_device_mismatch', 'risk_tier']])
print(f'\nDevice mismatches: {kyc["has_device_mismatch"].sum()} of {len(kyc)}')
print(f'New accounts (<30 days): {(kyc["account_age_days"] < 30).sum()}')
print(f'\nRisk tier distribution:')
print(kyc['risk_tier'].value_counts())

## 6. Alert rule preview — which transactions would be flagged?

This previews the logic we'll build properly in Week 2 (the alert engine).

In [ ]:
HIGH_RISK_COUNTRIES = {'IR', 'KP', 'SY', 'MM', 'YE', 'SD'}
HIGH_VALUE_THRESHOLD = 40_000

# Rule 1: High value (above AED 40k reporting threshold)
rule_high_value = txs['amount_aed'] > HIGH_VALUE_THRESHOLD

# Rule 2: Sanctioned corridor
rule_sanctioned = txs['country'].isin(HIGH_RISK_COUNTRIES)

# Combined: would trigger at least one alert
any_rule = rule_high_value | rule_sanctioned

print('=== Alert rule preview (Week 2 logic) ===')
print(f'High-value (> AED 40k):     {rule_high_value.sum()} transactions')
print(f'Sanctioned corridor:         {rule_sanctioned.sum()} transactions')
print(f'Would trigger an alert:      {any_rule.sum()} transactions')
print(f'True positives (real fraud): {(any_rule & txs["is_flagged"]).sum()}')
print(f'False positives:             {(any_rule & ~txs["is_flagged"]).sum()}')

precision = (any_rule & txs['is_flagged']).sum() / any_rule.sum() if any_rule.sum() > 0 else 0
recall = (any_rule & txs['is_flagged']).sum() / txs['is_flagged'].sum()
print(f'\nRule engine precision: {precision:.1%}')
print(f'Rule engine recall:    {recall:.1%}')
print('\n(These improve when agents add KYC + sanctions matching in Week 2+)')

## 7. Sanctions watchlist preview

In [ ]:
print('=== Sanctions Watchlist ===')
for entry in raw_sanctions:
    print(f"\n🚫 {entry['name']}")
    print(f"   Country: {entry['country']} | Reason: {entry['reason']}")
    print(f"   Aliases: {', '.join(entry['aliases'])}")

---

## Week 1 summary

| Component | Status |
|---|---|
| Synthetic transactions dataset | ✅ 50 records, 20% fraud rate |
| Amount distribution | ✅ Clear separation normal vs suspicious |
| Country risk profiling | ✅ High-risk corridors visible in suspicious txs |
| Time-of-day pattern | ✅ Suspicious txs cluster at 2–5am |
| KYC profiles | ✅ 10 customers, device mismatch signals |
| Sanctions watchlist | ✅ 5 entities with Arabic name variants |
| Rule engine preview | ✅ High precision, needs agent refinement |

### Next: Week 2
Build the **FastAPI alert engine** with proper Pydantic models, deterministic rules, and `POST /v1/alerts` endpoints.